In [ ]:
#avoid TF to consume GPU memory
import os
import glob
import tensorflow as tf
import mdtraj as md
import matplotlib.pyplot as plt
import matplotlib as mpl
import gromacs as gmx 
import numpy as np
import nglview as nv
import ASMSAnalysis 

tf.config.set_visible_devices([], 'GPU')
tf.config.list_logical_devices()

#%matplotlib widget


In [ ]:
conf = "../../dataset_Thermal-unfolding/trpcage_npt400_nH.pdb"
traj = "../../dataset_Thermal-unfolding/trpcage_ds_nH.xtc"
hills = "HILLS"
lows_filename = "../lows.txt"

## On the Fly Analysis

The “on-the-fly” module allows you to monitor and track the progress of the enhanced simulation in the latent space.
The circle highlights the reference structure, while the cross indicates the current position.

In [ ]:
analysis = ASMSAnalysis.Analysis(conf,traj,hills,lows_filename)
analysis.on_the_flight(interval=30)

## RMSD and Native contact fraction

The "calculate_rmsd" module will return the Root Mean Square Deviation (RMSD) of the simulation considering the first frame as reference. 
You can costum the index, allowing to follow the desired motion.

The "native_contact" module [...] not yet implemented. The code in the cell will provide the desired plot with the rmsd plot on its side

In [ ]:
idx_res = tr.top.select("residue 6")
idx_b = tr.top.select("backbone")
idx_ca = tr.top.select("name CA")
idx = tr.top.select("protein")

rmsd = analysis.calculate_rmsd(index=idx_b)

In [ ]:
traj_path = 'MD_fit.xtc'
top_path  = 'npt_nh.pdb'
ref_path  = top_path
scheme    = 'closest-heavy'
cutoff    = 0.50  # nm (5.0 Å)

# -------------------- STYLE --------------------
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.linewidth": 1.2,
    "axes.edgecolor": "#333333",
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.color": "#333333",
    "ytick.color": "#333333",
})

def add_panel_label(ax, label, dx=0.0, dy=0.02):
    ax.text(0.0 + dx, 1.0 + dy, label, transform=ax.transAxes,
            ha="left", va="bottom", fontsize=14, weight="bold", color="#222")

# -------------------- LOAD & ALIGN --------------------
traj = md.load(traj_path, top=top_path)
ref  = md.load(ref_path)[0]
idx_backbone = traj.top.select("backbone")
traj.superpose(traj, 0, atom_indices=idx_backbone)

t_ns = traj.time / 1000.0
rmsd = md.rmsd(traj, traj, 0)

# -------------------- ROBUST CONTACTS FUNCTION --------------------
def get_pairs_and_dists(traj_or_ref, *, scheme=None, contacts=None):
    if contacts is None:
        a, b = md.compute_contacts(traj_or_ref, scheme=scheme)
    else:
        a, b = md.compute_contacts(traj_or_ref, contacts=contacts, scheme=scheme)
    def _is_pairs(x):
        return isinstance(x, np.ndarray) and x.ndim == 2 and x.shape[1] == 2 and np.issubdtype(x.dtype, np.integer)
    if _is_pairs(a):
        pairs, dists = a, b
    elif _is_pairs(b):
        pairs, dists = b, a
    else:
        raise ValueError(f"Cannot identify pair matrix: {getattr(a,'shape',None)} vs {getattr(b,'shape',None)}")
    return np.asarray(pairs, dtype=int), np.asarray(dists)

# -------------------- COMPUTE Q --------------------
pairs_all, d_ref = get_pairs_and_dists(ref, scheme=scheme)
d_ref = np.squeeze(d_ref)
native_pairs = pairs_all[d_ref < cutoff]
if native_pairs.size == 0:
    raise RuntimeError(f"No native contacts found (cutoff={cutoff:.2f} nm). Try cutoff=0.55 nm or scheme='ca'.")
_, d_traj = get_pairs_and_dists(traj, scheme=scheme, contacts=native_pairs)
Q = (d_traj < cutoff).mean(axis=1)

# -------------------- FIGURE (1×2, WIDE) --------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), constrained_layout=True)

# (A) RMSD over time
ax = axes[0]
sc = ax.scatter(t_ns, rmsd, c=rmsd, s=4, cmap="coolwarm_r", rasterized=True, edgecolors='none')
ax.set_title(f"RMSD")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("RMSD (nm)")
add_panel_label(ax, "A")
cbar = fig.colorbar(sc, ax=ax, pad=0.02, shrink=0.95)
cbar.set_label("RMSD (nm)")

# (B) Q over time
ax = axes[1]
ax.plot(t_ns, Q, lw=1.4, color="#2B7BBC")
ax.set_title(f"Native-contact fraction")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Q (0–1)")
ax.set_ylim(-0.02, 1.02)
#ax.grid(True, ls="--", lw=0.6, alpha=0.6)
add_panel_label(ax, "B")

# -------------------- SAVE & SHOW --------------------
plt.savefig("rmsd_and_Q.png", dpi=300, bbox_inches="tight")
plt.show()


## 3D Visualization

The module "highlights_and_dynamic" will allow you to analyze the defined 3D structure within its position in the latent space. 

By moving the 3D representation of the system, the latent space will be uploaded accordingly to its exploration. The cross will show the point relative to the current 3D representation 

In [ ]:
analysis.highlights_and_dynamic()

In [ ]:
conf_nh, traj_fit = analysis.prepare_trajectory()
tr = md.load(traj_fit, top=conf_nh)

directory = './'  
files_to_delete = glob.glob(os.path.join(directory, '#*'))
for file in files_to_delete:
    os.remove(file)

v = nv.show_mdtraj(tr, default=False, gui=True)  # Enable GUI for play button
v.add_cartoon(selection="protein", color='red')
c = v.add_component(tr[0], default=False, gui=False)
c.add_cartoon(selection="protein", color='blue')

v.add_licorice(selection="res 6", c='red')
c.add_licorice(selection="res 6", c='blue')


v.center(selection='protein')
v.render_image(trim=True, factor=3)
v.camera = 'orthographic'
v.background = 'white'
v

In [ ]:
#gmx.editconf(f='npt.gro',input='Protein-H',o='npt_nh.pdb', ndef=True)

In [ ]:
directory = './'  
files_to_delete = glob.glob(os.path.join(directory, '#*'))
for file in files_to_delete:
    os.remove(file)

## Free Energy Calculations w/ Metadynminer

For more informations:
* Manuscript: https://academic.oup.com/bioinformatics/article/40/10/btae614/7826610
* GitHub Repo: https://github.com/Jan8be/metadynminer.py

In [ ]:
!pip install metadynminer

In [ ]:
import metadynminer as m

In [ ]:
hf = m.Hills(name="../lows.txt")

In [ ]:
fes = m.Fes(hf)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import mdtraj as md


In [ ]:
tr = md.load('../00.computations/x_test.xtc',top='../../dataset_Thermal-unfolding/trpcage_npt400_nH.pdb')
base = md.load('../../dataset_Thermal-unfolding/trpcage_npt400_nH.pdb')
m = torch.jit.load('../03.model/model.pt')
l = m(torch.tensor(tr.xyz)).numpy()
l1,l2 = l[:,0],l[:,1]

In [ ]:

plt.figure(figsize=(8, 6))
plt.hist2d(l1, l2, bins=200, cmap='plasma')
plt.colorbar(label='Density')
plt.xlabel('l1')
plt.ylabel('l2')
plt.title('Density Plot of l1 vs l2')
plt.show()

In [ ]:
fes.plot()

In [ ]:
minima = m.Minima(fes)
print(minima.minima)
minima.plot()

In [ ]:
fep = metadynminer.FEProfile(minima, hillsfile)
fep.plot()